In [2]:

# Everything we need in one cell

from google import genai
from google.genai.types import Part
import pandas as pd
from pydantic import BaseModel

from cred import GEMINI_KEY


def get_gemini_sentiment(api_key: str,
                          instructions: str,
                            texts_column: pd.Series,
                              schema: BaseModel,
                                temperature: float = 0.0) -> pd.DataFrame:
    
    api = genai.Client(api_key=api_key)
    data_json = texts_column.to_json()
    prompt = [Part(text=instructions), Part(text=data_json)]
    config={'response_mime_type': 'application/json',
             'response_schema': list[schema],
               'temperature': temperature}
    response = api.models.generate_content(model="gemini-2.5-flash", contents=prompt, config=config)
    results_table = pd.DataFrame([dict(record) for record in response.parsed])
    results_table = results_table.set_index('index')
    return results_table


import pandas as pd

# Define the nlp model

# Load in our dataset
articles = pd.read_parquet('farright_dataset_cleaned.parquet').head(100)
articles.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  100 non-null    object             
 1   type                100 non-null    object             
 2   sectionId           100 non-null    object             
 3   sectionName         100 non-null    object             
 4   webPublicationDate  100 non-null    datetime64[ns, UTC]
 5   webTitle            100 non-null    object             
 6   webUrl              100 non-null    object             
 7   apiUrl              100 non-null    object             
 8   tags                100 non-null    object             
 9   isHosted            100 non-null    bool               
 10  pillarId            100 non-null    object             
 11  pillarName          100 non-null    object             
 12  byline              100 non-null    o

In [5]:
articles['paragraph'] = articles['cleaned_text'].str.split('\n')

In [6]:
per_para = articles.explode('paragraph')
per_para = per_para[['paragraph']].reset_index(drop=False, names='article_idx')


In [7]:
per_para

,article_idx,paragraph
0,0,Senior Labour MPs and the UK's largest anti-fa...
1,0,Hope Not Hate's chief executive has written a ...
2,0,"In the letter, Nick Lowles said: “Hate breeds ..."
3,0,The challenge to No 10 speaks to many MPs' fea...
4,0,MPs who have called for greater clarity in the...
...,...,...
2258,99,"At protests, Bolsonaristas wield placards clai..."
2259,99,"Sebastião Coelho, a retired Bolsonarista judge..."
2260,99,Officials in Donald Trump's administration hav...
2261,99,Those who have tracked Moraes's career doubt s...


In [13]:
instructions = """You are a specialist in named entity recogntion. When given a set of texts you are able to identify names of individuals and organisations and use context to disambiguate when only first or surnames are used. I have provided you a set of JSON formatted documents. Each document has an index number and a paragraph of text. Analyse each paragraph and return a JSON formatted response containing the id number of the paragraph and a comma seperated set of unique entities found in it. Return a JSON formatted response"""

class NER(BaseModel):
    index:int
    entities:str




results = get_gemini_sentiment(api_key=GEMINI_KEY, instructions=instructions, texts_column=per_para['paragraph'].head(20), schema=NER)

results

,entities
index,
0,"Labour, Keir Starmer, Nigel Farage"
1,"Hope Not Hate, Keir Starmer, London"
2,Nick Lowles
3,"No 10, Nigel Farage, Labour, Bridget Phillipso..."
4,
5,"Louise Haigh, UK"
6,Nigel Farage
7,"Alison McGovern, Labour"
8,"Labour, British"


0    Senior Labour MPs and the UK's largest anti-fa...
1    Hope Not Hate's chief executive has written a ...
2    In the letter, Nick Lowles said: “Hate breeds ...
3    The challenge to No 10 speaks to many MPs' fea...
4    MPs who have called for greater clarity in the...
5    The former cabinet minister Louise Haigh said:...
Name: paragraph, dtype: object